# Question 3: Dog Image Inpainting - Segmentation-Guided, SAM (Demo)

This notebook demonstrates how the Segment Anything Model (SAM) can be used to generate realistic object masks for foreground occlusions. We explored SAM as an additional advanced step to make the masking process more realistic than simple synthetic rectangle masks. This approach better simulates real-world occlusions such as leaves, poles, or other foreground objects blocking parts of an image. SAM is used for mask generation not image generation and the actual image completion can then be performed by U-Net, edge-guided U-Net, or Stable Diffusion inpainting.


**Goal:** Given a dog image with a foreground occlusion, such as a pole, fence, leaves, or branches, generate an occlusion mask and use an inpainting model to remove the blocker and restore the image.

This notebook is organized around the new pipeline:

1. Load occluded dog images  
2. Create or load masks for blocking objects  
3. Run pretrained inpainting models  
4. Compare outputs with the U-Net/random-mask baseline  
5. Later extension: automatic mask generation with segmentation


## 1. Environment Setup

Use a GPU runtime in Colab:  
`Runtime → Change runtime type → GPU`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Core imports
import os
from pathlib import Path
import random

import numpy as np
from PIL import Image, ImageOps
import matplotlib.pyplot as plt

import torch
from torchvision import transforms

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# Install SAM dependencies in Colab
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q opencv-python pycocotools matplotlib onnxruntime

In [ ]:
# Download SAM checkpoint - Use the ViT-B model first (smaller/faster)
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

In [ ]:
# Load SAM
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
import cv2
import matplotlib.pyplot as plt

sam_checkpoint = "sam_vit_b_01ec64.pth"
model_type = "vit_b"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device="cuda")

mask_generator = SamAutomaticMaskGenerator(sam)

## 2. Project Folders

This creates a clean structure for the new pipeline.


In [ ]:
CANDIDATE_DIRS = [
    '/content/drive/Shareddrives/Adv CV/Computer Vision Group 3/Q3 outputs',
    '/content/drive/MyDrive/Adv CV/Computer Vision Group 3/Q3 outputs',
    '/content/drive/MyDrive/Adv CV/computer vision group 3/Q3 outputs',
]

BASE_DIR = None
for d in CANDIDATE_DIRS:
    if os.path.exists(os.path.dirname(d)):
        BASE_DIR = d
        break

if BASE_DIR is None:
    BASE_DIR = CANDIDATE_DIRS[0]

IMAGE_DIR  = Path(BASE_DIR) / 'occlusion_images'
MASK_DIR   = Path(BASE_DIR) / 'masks'
RESULT_DIR = Path(BASE_DIR) / 'inpainting_results'
MODEL_DIR  = Path(BASE_DIR) / 'models'

for p in [IMAGE_DIR, MASK_DIR, RESULT_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('BASE_DIR:', BASE_DIR)
print('IMAGE_DIR:', IMAGE_DIR)
print('MASK_DIR:', MASK_DIR)
print('RESULT_DIR:', RESULT_DIR)
print('MODEL_DIR:', MODEL_DIR)

## 3. Load Occluded Dog Images

Put a few occluded dog images into:

`Q3_outputs/occlusion_images/`

Start with 5–10 images. The goal is to make the pipeline work first, then scale up.


In [ ]:
IMAGE_DIR = "/content/drive/MyDrive/Adv CV/CV Project Folder/sample_occlusion_images"

In [ ]:
def list_images(folder):
    exts = {'.jpg', '.jpeg', '.png', '.webp'}
    return sorted([p for p in Path(folder).rglob('*') if p.suffix.lower() in exts])

image_paths = list_images(IMAGE_DIR)
print(f'Found {len(image_paths)} occlusion images')
for p in image_paths[:10]:
    print(p.name)

In [ ]:
# Preview images
if len(image_paths) > 0:
    n = min(len(image_paths), 6)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1:
        axes = [axes]
    for ax, path in zip(axes, image_paths[:n]):
        img = Image.open(path).convert('RGB')
        ax.imshow(img)
        ax.set_title(path.name, fontsize=9)
        ax.axis('off')
    plt.tight_layout()
else:
    print('No images found yet. Upload images into IMAGE_DIR and rerun this cell.')

## 4. Test on One Image

In [ ]:
image_path = image_paths[0]

image = cv2.imread(str(image_path))
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

masks = mask_generator.generate(image)

print(f"Generated {len(masks)} masks")

Visualize masks:

In [ ]:
plt.figure(figsize=(8,8))
plt.imshow(image)

for mask in masks:
    plt.contour(mask['segmentation'], colors='red', linewidths=0.5)

plt.axis('off')
plt.show()

## 5. Select foreground occluder mask

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(image)

for i, mask in enumerate(masks):

    seg = mask['segmentation']

    plt.contour(seg, colors='red', linewidths=1)

    y, x = np.argwhere(seg).mean(axis=0)

    plt.text(
        x, y,
        str(i),
        color='yellow',
        fontsize=12,
        bbox=dict(facecolor='black', alpha=0.5)
    )

plt.axis('off')
plt.show()

In [ ]:
selected_ids = [0, 1, 3, 9] # select slightly more areas

combined_mask = np.zeros(image.shape[:2], dtype=bool)

for idx in selected_ids:
    combined_mask = combined_mask | masks[idx]["segmentation"]

plt.figure(figsize=(6,6))
plt.imshow(combined_mask, cmap="gray")
plt.title("Combined occlusion mask")
plt.axis("off")
plt.show()

Here, SAM generated candidate segmentation masks, and we selected/combined masks corresponding to foreground occluders before applying inpainting.

Overlay it on the original image:

In [ ]:
import cv2
import numpy as np

kernel = np.ones((9, 9), np.uint8)

dilated_mask = cv2.dilate(
    combined_mask.astype(np.uint8),
    kernel,
    iterations=1
)

plt.figure(figsize=(8,8))
plt.imshow(image)
plt.imshow(dilated_mask, alpha=0.5, cmap="Reds")
plt.title("Dilated improved mask")
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(8,8))
plt.imshow(image)
plt.imshow(combined_mask, alpha=0.45, cmap="Reds")
plt.title("SAM occlusion mask overlay")
plt.axis("off")
plt.show()

In this section, we found that better segmentation masks directly improve downstream generative restoration quality.

Save the mask:

In [ ]:
from PIL import Image
import numpy as np

mask_uint8 = (combined_mask.astype(np.uint8) * 255)
mask_pil = Image.fromarray(mask_uint8)

mask_path = f"{MASK_DIR}/{image_path.stem}_sam_mask.png"
mask_pil.save(mask_path)

print("Saved mask to:", mask_path)

# 6. Run Demo Inpainting with Stable Diffusion Model (Generative Inpainting)


In [ ]:
from diffusers import StableDiffusionInpaintPipeline
import torch
from PIL import Image

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=torch.float16
).to("cuda")

init_image = Image.open(image_path).convert("RGB").resize((512, 512))
mask_image = Image.open(mask_path).convert("L").resize((512, 512))

result = pipe(
    prompt="a realistic photo of a dog, natural fur, high quality",
    image=init_image,
    mask_image=mask_image
).images[0]

result.save(f"{RESULT_DIR}/{image_path.stem}_inpainted.png")
result

In [ ]:
from diffusers import StableDiffusionInpaintPipeline
import torch
from PIL import Image

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=torch.float16
).to("cuda")

init_image = Image.open(image_path).convert("RGB").resize((512, 512))
mask_image = Image.open(mask_path).convert("L").resize((512, 512))

result = pipe(
    prompt = """a happy white fluffy dog, complete visible forehead, natural fur texture, cute smiling dog portrait, professional pet photo""",
    image=init_image,
    mask_image=mask_image
).images[0]

result.save(f"{RESULT_DIR}/{image_path.stem}_inpainted.png")
result

### Potential Prompts to guide Diffusion Model: improve the model with prompt engineering

In [ ]:
prompt = """
a realistic fluffy white dog face,
natural dog fur,
symmetrical dog head,
high quality pet photography,
outdoor lighting
"""

negative_prompt = """
leaf, plant, cloth, fabric, hat,
distorted face, extra object,
deformed, blurry
"""

prompt = """
a happy white fluffy dog,
complete visible forehead,
natural fur texture,
cute smiling dog portrait,
professional pet photo
"""